In [3]:
import torch
from sentence_transformers import SentenceTransformer, util

MODEL_NAME = "all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(MODEL_NAME, device=device)
device

'cuda'

#### corpus

In [4]:
corpus = [
    "Patient has a history of COPD and requires regular inhaler use.",
    "ECG shows ST elevation consistent with myocardial infarction.",
    "Blood tests indicate elevated CRP suggesting infection.",
    "MRI scan reveals no acute intracranial abnormalities.",
    "Patient presents with shortness of breath and chest tightness."
]

corpus_embeddings = model.encode(corpus, convert_to_tensor=True)
corpus_embeddings.shape


torch.Size([5, 384])

#### Simple RAG‑like retrieval function

In [5]:
def retrieve(question: str, top_k: int = 3):
    q_emb = model.encode(question, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, corpus_embeddings)[0]
    top_results = torch.topk(scores, k=top_k)
    return [
        (corpus[idx], scores[idx].item())
        for idx in top_results.indices
    ]


#### try some clinical queries

In [6]:
questions = [
    "Which note suggests infection?",
    "Which note is about heart attack?",
    "Which note mentions breathing problems?"
]

for q in questions:
    print(f"\nQuery: {q}")
    for text, score in retrieve(q, top_k=2):
        print(f"  Score: {score:.4f} | {text}")



Query: Which note suggests infection?
  Score: 0.4023 | Blood tests indicate elevated CRP suggesting infection.
  Score: 0.2348 | Patient presents with shortness of breath and chest tightness.

Query: Which note is about heart attack?
  Score: 0.3279 | Patient presents with shortness of breath and chest tightness.
  Score: 0.3089 | ECG shows ST elevation consistent with myocardial infarction.

Query: Which note mentions breathing problems?
  Score: 0.4955 | Patient presents with shortness of breath and chest tightness.
  Score: 0.4106 | Patient has a history of COPD and requires regular inhaler use.
